<a href="https://colab.research.google.com/github/Nayab189/flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

# Setup & Data Ingestion (with Full-Dataset Retrain for Production Scoring)

In [45]:
import os
import duckdb
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Create required output folders
os.makedirs('../work/outputs', exist_ok=True)
os.makedirs('../work/figures', exist_ok=True)

# 1. Connect DuckDB & Authenticate HF Secret
con = duckdb.connect()
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
except Exception as e:
    print("Token Note:", e)

rel = "hf://datasets/FlyRank/internship-warehouse"

# 2. Query Warehouse Data with Strict March->April Temporal Split, Real Content Age & CTR Feature
df = con.sql(f"""
    WITH feature_window AS (
        SELECT
            p.content_hash_id AS content_id,
            SUM(p.gsc_impressions) AS impressions_30d,
            SUM(p.gsc_clicks) AS clicks_30d,
            CASE
                WHEN SUM(p.gsc_impressions) > 0 THEN (SUM(p.gsc_clicks) * 1.0 / SUM(p.gsc_impressions))
                ELSE 0.0
            END AS ctr_30d,
            CASE
                WHEN SUM(p.gsc_impressions) > 0 THEN LEAST(100.0, (SUM(p.gsc_sum_position) * 1.0 / SUM(p.gsc_impressions)))
                ELSE 100.0
            END AS avg_position,
            COALESCE(DATE_DIFF('day', CAST(d.content_created_date AS DATE), DATE '2026-03-31'), 90) AS content_age_days
        FROM read_parquet('{rel}/fact_content_daily_performance/*/*.parquet') p
        LEFT JOIN read_parquet('{rel}/dim_content.parquet') d ON p.content_hash_id = d.content_hash_id
        WHERE p.report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY p.content_hash_id, d.content_created_date
    ),
    label_window AS (
        SELECT
            content_hash_id AS content_id,
            SUM(gsc_impressions) AS future_impressions_30d
        FROM read_parquet('{rel}/fact_content_daily_performance/*/*.parquet')
        WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
        GROUP BY content_hash_id
    )
    SELECT
        f.*,
        CASE
            WHEN l.content_id IS NULL THEN 1
            WHEN l.future_impressions_30d < (f.impressions_30d * 0.8) THEN 1
            ELSE 0
        END AS is_declining
    FROM feature_window f
    LEFT JOIN label_window l ON f.content_id = l.content_id
   ORDER BY f.content_id
    LIMIT 100000
""").df()

# Handle missing values & derived features
df['impressions_30d'] = df['impressions_30d'].fillna(0.0)
df['clicks_30d'] = df['clicks_30d'].fillna(0.0)
df['ctr_30d'] = df['ctr_30d'].fillna(0.0)
df['avg_position'] = df['avg_position'].fillna(100.0)
df['content_age_days'] = df['content_age_days'].fillna(90.0)
df['log_impressions_30d'] = np.log1p(df['impressions_30d'])

# 3. Model Training, Honest Holdout Validation & Full-Dataset Retraining (Including ctr_30d)
feature_cols = ['impressions_30d', 'log_impressions_30d', 'ctr_30d', 'avg_position', 'content_age_days']
X = df[feature_cols]
y = df['is_declining']

# Train/Test split for honest evaluation
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
eval_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
eval_model.fit(X_tr, y_tr)

train_auc = roc_auc_score(y_tr, eval_model.predict_proba(X_tr)[:, 1])
honest_auc = roc_auc_score(y_te, eval_model.predict_proba(X_te)[:, 1])

# Retrain on full dataset after honest validation for maximum deployment coverage
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X, y)
df['opportunity_score'] = rf_model.predict_proba(X)[:, 1]

print(f" SETUP COMPLETE! Rows: {len(df):,}")
print(f" Train ROC-AUC: {train_auc:.4f} | Honest Test ROC-AUC: {honest_auc:.4f}")
print(f" Class Balance (is_declining=1 ratio): {y.mean():.2%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 SETUP COMPLETE! Rows: 100,000
 Train ROC-AUC: 0.8899 | Honest Test ROC-AUC: 0.8902
 Class Balance (is_declining=1 ratio): 28.38%


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

To turn continuous model outputs into actionable human workflows, we map raw risk scores and page search metrics into explicit **Content Archetypes** and **Rule-Based Reason Codes**.

> **Capstone Research Alignment Note:** Per the Week 5–6 ablation findings, AI-referral session features were explicitly excluded from this production playbook model, as they showed negligible incremental predictive lift (~0.0000–0.0005 ROC-AUC) over traditional performance signals.

**Action Decision Framework**
1. **P1 — Full Content Overhaul (`REASON_FULL_OVERHAUL`)**:
   - *Condition:* High Opportunity Score (>0.70), content age > 120 days, and declining trajectory.
   - *Action:* Complete rewrite, update outdated statistics, and add new subheadings addressing current search intent.
2. **P2 — CTR & Title Optimization (`REASON_CTR_TITLE_REFRESH`)**:
   - *Condition:* High Impressions (>1,000) and Page 2 average position (Rank 11–25).
   - *Action:* Test new meta titles/descriptions and improve search snippet appeal.
3. **P3 — Internal Linking Boost (`REASON_INTERNAL_LINK_BOOST`)**:
   - *Condition:* High position rank (>30) with low relative impressions (<100).
   - *Action:* Add 3–5 contextual internal links from top-performing pillar pages to pass authority.
4. **P4 — Monitor / Maintain (`REASON_MONITOR_ONLY`)**:
   - *Condition:* Low risk score or newly published content (<45 days old).
   - *Action:* Retain as-is; allow baseline data collection.

In [46]:
# Generating Archetypes, Reason Codes & Ranked Queue (Execution with Actionable-Only Top Queue)
def assign_action_playbook(row):
    if row['opportunity_score'] >= 0.70 and row['content_age_days'] > 120:
        return 'P1_FULL_OVERHAUL', 'REASON_DECAYED_HIGH_RISK_CONTENT'
    elif row['impressions_30d'] > 1000 and 11 <= row['avg_position'] <= 25:
        return 'P2_TITLE_CTR_REFRESH', 'REASON_STRIKING_DISTANCE_PAGE2'
    elif row['avg_position'] > 30 and row['impressions_30d'] < 100:
        return 'P3_INTERNAL_LINKING', 'REASON_LOW_AUTHORITY_POOR_RANK'
    else:
        return 'P4_MONITOR_ONLY', 'REASON_STABLE_OR_NEW_CONTENT'

# Apply archetype mapping
playbook_results = df.apply(assign_action_playbook, axis=1)
df['action_tier'] = [res[0] for res in playbook_results]
df['reason_code'] = [res[1] for res in playbook_results]

# Priority Score for Queue Ranking
df['priority_rank_score'] = df['opportunity_score'] * (np.log1p(df['impressions_30d']) + 1.0)
ranked_queue = df.sort_values(by='priority_rank_score', ascending=False)

# Exclude P4 from Top Action Items so editors only see actionable tasks
actionable_only = ranked_queue[ranked_queue['action_tier'] != 'P4_MONITOR_ONLY']

print("=== ACTION QUEUE DISTRIBUTION ===")
print(df['action_tier'].value_counts().to_string())

print("\n=== TOP 5 PRIORITY ACTION ITEMS (Actionable Tiers P1-P3 Only) ===")
print(actionable_only[['content_id', 'action_tier', 'reason_code', 'priority_rank_score', 'impressions_30d', 'avg_position']].head(5).to_string(index=False))

=== ACTION QUEUE DISTRIBUTION ===
action_tier
P3_INTERNAL_LINKING     51369
P4_MONITOR_ONLY         45195
P2_TITLE_CTR_REFRESH     2392
P1_FULL_OVERHAUL         1044

=== TOP 5 PRIORITY ACTION ITEMS (Actionable Tiers P1-P3 Only) ===
              content_id          action_tier                    reason_code  priority_rank_score  impressions_30d  avg_position
content_3df3f32f3fd58dea P2_TITLE_CTR_REFRESH REASON_STRIKING_DISTANCE_PAGE2             7.286345         140156.0     23.591997
content_02474d1ff0eca1e5 P2_TITLE_CTR_REFRESH REASON_STRIKING_DISTANCE_PAGE2             6.998194          73019.0     21.982621
content_24c5fe46996aee20 P2_TITLE_CTR_REFRESH REASON_STRIKING_DISTANCE_PAGE2             6.963100          38052.0     15.305871
content_48eb41a492b8a15d P2_TITLE_CTR_REFRESH REASON_STRIKING_DISTANCE_PAGE2             6.942521          66598.0     24.917505
content_2f09787bdf392b16 P2_TITLE_CTR_REFRESH REASON_STRIKING_DISTANCE_PAGE2             6.852764          56233.0     23.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Operating Scope**
- **Decision-Support Filter:** This playbook serves strictly as a **prioritization engine** for content editorial teams to filter thousands of catalog pages down to top high-leverage refresh targets.
- **Directional Opportunity Mapping:** Scores quantify relative risk of traffic loss under historical decay trends rather than guaranteeing exact organic revenue or traffic outcomes.

**System Boundaries & Limitations**
- **Seasonality Blind Spots:** The model operates on short-term window snapshots. Seasonal queries may be over-flagged during off-peak periods.
- **No Algorithmic Penalty Awareness:** The model cannot detect Google manual actions or core algorithm update shifts directly without external search console integration.

**Cost / Value Resource Allocation Matrix**
| Priority Tier | Estimated Editorial Effort | Expected Business Impact | Cost / Value Rationale |
| :--- | :--- | :--- | :--- |
| **P1 (Full Overhaul)** | High (4–6 hours/page) | High (Traffic recovery) | Reserve for high-visibility decaying assets where full rewrites yield maximum ROI. |
| **P2 (Title/CTR Refresh)** | Low (30 mins/page) | Moderate-High (Quick wins) | Highly efficient; targets striking-distance keywords with minimal text changes. |
| **P3 (Internal Linking)** | Low (15 mins/page) | Moderate (Authority boost) | Automated/semi-automated link placement passes PageRank efficiently. |
| **P4 (Monitor Only)** | Zero | Baseline maintenance | Avoids wasting human resources on stable or brand-new content. |

In [47]:
# Limit & Boundary Check Summary
limit_audit = {
    "total_catalog_pages": len(df),
    "actionable_p1_p2_pages": int((df['action_tier'].isin(['P1_FULL_OVERHAUL', 'P2_TITLE_CTR_REFRESH'])).sum()),
    "low_signal_young_pages": int((df['content_age_days'] < 45).sum()),
    "honest_model_auc": round(float(honest_auc), 4)
}

print("=== INTENDED USE & BOUNDARY METRICS ===")
print(json.dumps(limit_audit, indent=2))

=== INTENDED USE & BOUNDARY METRICS ===
{
  "total_catalog_pages": 100000,
  "actionable_p1_p2_pages": 3436,
  "low_signal_young_pages": 11905,
  "honest_model_auc": 0.8902
}


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Pre-Action Check Protocol**

Content managers must perform the following 3-point inspection prior to approving any AI-ranked recommendation:
1. **Search Intent Verification:** Search top-3 live Google SERPs to confirm if search intent shifted from informational to commercial.
2. **Canonical & URL Integrity:** Verify page is not a canonical duplicate or recently redirected landing page.
3. **Brand Compliance:** Ensure updated content preserves brand voice guidelines.

---

**THE NO-GO LIST (Strictly Prohibited from Direct Auto-Execution)**
- **Automated Article Overhauls:** Generative AI must **NEVER** auto-rewrite and auto-publish P1 overhaul pages without editorial sign-off.
- **Automated URL / Slug Changes:** Slugs or permalinks must **NEVER** be updated programmatically (prevents broken 404 links and loss of link equity).
- **Deletion / Unpublishing of High-Impression Pages:** Content deprecation requires legal and business owner approval.
- **Automated De-indexing (Noindex tags):** No automated script is allowed to add meta-noindex tags.

In [48]:
# Flagging No-Go Risks in Queue
df['no_go_flag'] = np.where(
    (df['impressions_30d'] > 10000) | (df['content_age_days'] < 30),
    'REQUIRES_EXECUTIVE_EDITORIAL_REVIEW',
    'STANDARD_EDITORIAL_REVIEW'
)

ranked_queue = df.sort_values(by='priority_rank_score', ascending=False)

print("=== HUMAN REVIEW SAFETY GOVERNANCE ===")
print(df['no_go_flag'].value_counts().to_string())

=== HUMAN REVIEW SAFETY GOVERNANCE ===
no_go_flag
STANDARD_EDITORIAL_REVIEW              89579
REQUIRES_EXECUTIVE_EDITORIAL_REVIEW    10421


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

To prevent recommendation drift and stale decision outputs, the system operates under a dual-trigger monitoring framework.

**Retrain & Alert Triggers**
1. **Model Performance Degradation:**
   - *Trigger:* If the rolling test ROC-AUC drops below **0.65** on new monthly performance snapshots.
2. **Distributional Feature Drift (PSI / KS-Test):**
   - *Trigger:* If monthly average impression or position distributions shift by $> 20\%$ compared to baseline distribution.
3. **Action Efficiency Decay:**
   - *Trigger:* If $< 50\%$ of recommended P1/P2 refreshes show positive directional impression recovery within 60 days post-update.

In [49]:
# Monitoring Metric Calculation & Baseline Health Check with Sanity Checks
baseline_drift_metric = float(np.abs(df['avg_position'].mean() - 48.5) / 48.5)

retrain_status = {
    "train_auc": round(float(train_auc), 4),
    "honest_test_auc": round(float(honest_auc), 4),
    "auc_overfitting_gap": round(float(train_auc - honest_auc), 4),
    "auc_threshold": 0.65,
    "feature_drift_ratio": round(baseline_drift_metric, 4),
    "drift_threshold": 0.20,
    "retrain_recommended": bool(honest_auc < 0.65 or baseline_drift_metric > 0.20)
}

print("=== MONITORING & SANITY CHECK STATUS ===")
print(json.dumps(retrain_status, indent=2))

=== MONITORING & SANITY CHECK STATUS ===
{
  "train_auc": 0.8899,
  "honest_test_auc": 0.8902,
  "auc_overfitting_gap": -0.0003,
  "auc_threshold": 0.65,
  "feature_drift_ratio": 0.1375,
  "drift_threshold": 0.2,
  "retrain_recommended": false
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

We export the finalized prioritized action queue to `work/outputs/action_queue.csv` and summary metrics to `work/outputs/playbook_metrics.json`.

Additionally, we render and export the **Archetype Priority Distribution** chart to `work/figures/playbook_queue_distribution.png` for direct inclusion in next week's capstone research paper.

In [50]:
if 'no_go_flag' not in df.columns:
    df['no_go_flag'] = np.where(
        (df['impressions_30d'] > 10000) | (df['content_age_days'] < 30),
        'REQUIRES_EXECUTIVE_EDITORIAL_REVIEW',
        'STANDARD_EDITORIAL_REVIEW'
    )
    ranked_queue = df.sort_values(by='priority_rank_score', ascending=False)

# 1. Export Action Queue CSV to work/outputs/
queue_export_path = '../work/outputs/action_queue.csv'
export_cols = ['content_id', 'action_tier', 'reason_code', 'priority_rank_score', 'opportunity_score', 'impressions_30d', 'avg_position', 'no_go_flag']
ranked_queue[export_cols].head(5000).to_csv(queue_export_path, index=False)
print(f"Exported Queue CSV to: {queue_export_path}")

# 2. Export Metrics JSON to work/outputs/
metrics_payload = {
    "total_analyzed_assets": len(df),
    "actionable_queue_size": int(len(ranked_queue[ranked_queue['action_tier'] != 'P4_MONITOR_ONLY'])),
    "tier_distribution": df['action_tier'].value_counts().to_dict(),
    "no_go_flag_distribution": df['no_go_flag'].value_counts().to_dict(),
    "train_auc": round(float(train_auc), 4),
    "honest_model_auc": round(float(honest_auc), 4)
}

metrics_export_path = '../work/outputs/playbook_metrics.json'
with open(metrics_export_path, 'w') as f:
    json.dump(metrics_payload, f, indent=2)
print(f"Exported Metrics JSON to: {metrics_export_path}")

# 3. Create & Export Visualization to work/figures/
plt.figure(figsize=(9, 5))
tier_counts = df['action_tier'].value_counts()
bars = plt.bar(tier_counts.index, tier_counts.values, color=['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c'])
plt.title('Content Action Playbook Queue Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Action Priority Tier', fontsize=10)
plt.ylabel('Number of Pages', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 500, f'{int(yval):,}', ha='center', va='bottom', fontsize=9)

figure_export_path = '../work/figures/playbook_queue_distribution.png'
plt.tight_layout()
plt.savefig(figure_export_path, dpi=300)
plt.close()

print(f"Exported Figure PNG to: {figure_export_path}")

Exported Queue CSV to: ../work/outputs/action_queue.csv
Exported Metrics JSON to: ../work/outputs/playbook_metrics.json
Exported Figure PNG to: ../work/figures/playbook_queue_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.